# Melodic EDM Core V2 — Colab Pro inference

[Open this notebook in Colab after the branch is merged](https://colab.research.google.com/github/Bangchis/melodic-edm-training-pipeline/blob/main/notebooks/melodic_edm_core_v2_colab.ipynb)

This notebook installs the pinned ACE-Step source, downloads pinned XL-Base weights and a private V2 LoRA release, verifies checksums, generates one deterministic WAV, and validates 48 kHz stereo output. It performs inference only.

## 1. Before running

Sign in to the Google account that owns Colab Pro in this browser, then select **Runtime → Change runtime type → NVIDIA GPU**. In Colab Secrets (key icon), add a Hugging Face read token named `HF_TOKEN` and an OpenRouter key named `OPENROUTER_API_KEY`; enable notebook access for both. Do not paste a token into a cell. No Google password, cookie or OAuth token belongs on the Vast server. Consumer Colab Pro has no supported server-side job-submission CLI; this notebook runs in the browser-created Colab runtime.

In [ ]:
import shutil, subprocess, torch
assert torch.cuda.is_available(), 'No NVIDIA GPU. Change the Colab runtime type.'
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 1024**3
disk = shutil.disk_usage('/content')
print(f'GPU: {gpu.name} | VRAM: {vram_gib:.1f} GiB')
print(f'Disk free: {disk.free / 1024**3:.1f} GiB')
assert disk.free / 1024**3 >= 35, 'At least 35 GiB free disk is required.'
OFFLOAD_TO_CPU = vram_gib < 20
print('CPU offload:', OFFLOAD_TO_CPU)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg git
!python -m pip -q install 'huggingface_hub>=0.30' uv

In [ ]:
import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN and HF_TOKEN.startswith('hf_'), 'HF_TOKEN is missing from Colab Secrets.'
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
assert OPENROUTER_API_KEY, 'OPENROUTER_API_KEY is missing from Colab Secrets.'
os.environ['HF_TOKEN'] = HF_TOKEN
print('Hugging Face and OpenRouter keys loaded from Colab Secrets (values not displayed).')

## 2. Pin every upstream component

Wait until the private `Bangchis/melodic-edm-core-v2` release has been uploaded. This notebook resolves its current head to one immutable commit SHA, prints that SHA, and uses only that SHA for the rest of the run. Save the printed SHA with any result you want to reproduce.

In [ ]:
ACE_SOURCE_REVISION = '6d467e4b5081ccb0abf1ec1bf4fdf9051a2d34b0'
ACE_CORE_REVISION = '19671f406d603126926c1b7e2adc169acbcade22'
XL_BASE_REVISION = '220c1166efbdd9583eafcb12eb160594bbfcb241'
RELEASE_REPO = 'Bangchis/melodic-edm-core-v2'
from huggingface_hub import HfApi
RELEASE_REVISION = HfApi(token=HF_TOKEN).model_info(RELEASE_REPO).sha
assert RELEASE_REVISION and len(RELEASE_REVISION) == 40, 'Hugging Face did not return an immutable release SHA.'
print('Pinned V2 release revision:', RELEASE_REVISION)
PREFERRED_ADAPTER = 'final-all-data'  # automatically falls back to best-val preview

In [ ]:
from pathlib import Path
ACE_ROOT = Path('/content/ACE-Step-1.5')
if not ACE_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ace-step/ACE-Step-1.5.git', str(ACE_ROOT)], check=True)
subprocess.run(['git', '-C', str(ACE_ROOT), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(ACE_ROOT), 'checkout', '--detach', ACE_SOURCE_REVISION], check=True)
subprocess.run(['uv', 'sync'], cwd=ACE_ROOT, check=True)
print('ACE-Step source and environment ready at', ACE_SOURCE_REVISION)

In [ ]:
from huggingface_hub import snapshot_download
CHECKPOINT_ROOT = ACE_ROOT / 'checkpoints'
snapshot_download(
    repo_id='ACE-Step/Ace-Step1.5', revision=ACE_CORE_REVISION,
    local_dir=CHECKPOINT_ROOT, token=HF_TOKEN,
)
snapshot_download(
    repo_id='ACE-Step/acestep-v15-xl-base', revision=XL_BASE_REVISION,
    local_dir=CHECKPOINT_ROOT / 'acestep-v15-xl-base', token=HF_TOKEN,
)
print('Pinned ACE-Step core and XL-Base checkpoints downloaded.')

In [ ]:
RELEASE_DIR = Path('/content/melodic-edm-core-v2')
snapshot_download(
    repo_id=RELEASE_REPO, revision=RELEASE_REVISION,
    repo_type='model', local_dir=RELEASE_DIR, token=HF_TOKEN,
)
if (RELEASE_DIR / PREFERRED_ADAPTER / 'adapter_config.json').is_file():
    ADAPTER = PREFERRED_ADAPTER
elif (RELEASE_DIR / 'best-val' / 'adapter_config.json').is_file():
    ADAPTER = 'best-val'
else:
    raise FileNotFoundError('Neither final-all-data nor best-val exists in this release.')
assert (RELEASE_DIR / ADAPTER / 'adapter_config.json').is_file()
assert (RELEASE_DIR / ADAPTER / 'adapter_model.safetensors').is_file()
subprocess.run(['sha256sum', '-c', 'SHA256SUMS'], cwd=RELEASE_DIR, check=True)
print('Private V2 release downloaded and every packaged checksum passed. Adapter:', ADAPTER)

## 3. Enhance a free-form idea through OpenRouter

The notebook sends only your text idea and explicit musical conditions to an OpenRouter LLM. The LLM must return strict JSON with exactly five music-description fields; a local deterministic gate then rejects artist shortcuts, quality hype, malformed fields and captions outside 40–300 words before ACE-Step can run. The OpenRouter key never enters the prompt JSON, inference subprocess, GitHub or Hugging Face.

In [ ]:
import json, sys
sys.path.insert(0, str(RELEASE_DIR / 'scripts'))
from enhance_prompt_openrouter import enhance_prompt
USER_IDEA = ('Nhạc EDM Trung Hoa không lời, phiêu lưu và tươi sáng, có hook pipa dễ nhớ, '
             'dizi đối đáp, mở đầu không khí rồi build ngắn và drop mạnh.')
OPENROUTER_MODEL = '~google/gemini-flash-latest'  # set an exact slug to pin the enhancer model
EXPLICIT_CONDITIONS = {
    'bpm': 128,
    'keyscale': 'F# minor',
    'timesignature': '4',
    'sections': ['Intro', 'Theme', 'Build', 'Drop', 'Break', 'Final Drop', 'Outro'],
}
enhancement = enhance_prompt(
    USER_IDEA, OPENROUTER_API_KEY, model=OPENROUTER_MODEL,
    explicit_conditions=EXPLICIT_CONDITIONS,
)
music_conditions = enhancement['conditions']
print('OpenRouter resolved model:', enhancement['resolved_model'])
print('Enhanced caption:', music_conditions['caption'])
print('Word count:', music_conditions['word_count'])
custom_prompt = {
    'prompts': [{
        'id': 'colab_custom_01',
        'caption': music_conditions['caption'],
        'bpm': music_conditions['bpm'],
        'keyscale': music_conditions['keyscale'],
        'timesignature': music_conditions['timesignature'],
        'lyrics': music_conditions['lyrics'],
        'duration': 45, 'seed': 260718,
    }]
}
ENHANCEMENT_FILE = Path('/content/v2_prompt_enhancement.json')
ENHANCEMENT_FILE.write_text(json.dumps(enhancement, ensure_ascii=False, indent=2), encoding='utf-8')
PROMPT_FILE = Path('/content/v2_custom_prompt.json')
PROMPT_FILE.write_text(json.dumps(custom_prompt, ensure_ascii=False, indent=2), encoding='utf-8')
print('Conditioning payload ready:', json.dumps(custom_prompt, ensure_ascii=False, indent=2))

In [ ]:
OUTPUT_DIR = Path('/content/generated-v2')
command = [
    str(ACE_ROOT / '.venv/bin/python'),
    str(RELEASE_DIR / 'scripts/infer_v2_release.py'),
    '--ace-root', str(ACE_ROOT),
    '--checkpoint-root', str(CHECKPOINT_ROOT),
    '--release-dir', str(RELEASE_DIR),
    '--adapter-subdirectory', ADAPTER,
    '--prompts', str(PROMPT_FILE),
    '--prompt-index', '0',
    '--output-dir', str(OUTPUT_DIR),
]
if OFFLOAD_TO_CPU:
    command.append('--offload-to-cpu')
print('Starting ACE-Step inference. Full output is also saved to /content/v2_inference.log')
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ, 'HF_TOKEN': HF_TOKEN},
)
inference_log_lines = []
for line in process.stdout:
    print(line, end='')
    inference_log_lines.append(line)
return_code = process.wait()
INFERENCE_LOG = Path('/content/v2_inference.log')
INFERENCE_LOG.write_text(''.join(inference_log_lines), encoding='utf-8')
if return_code != 0:
    tail = ''.join(inference_log_lines[-120:])
    raise RuntimeError(
        f'ACE-Step inference exited with code {return_code}. Full log: {INFERENCE_LOG}\n'
        f'Last output lines:\n{tail}'
    )

In [ ]:
from IPython.display import Audio, display
report = json.loads((OUTPUT_DIR / 'inference_report.json').read_text(encoding='utf-8'))
assert report['status'] == 'pass'
assert report['probe']['sample_rate'] == 48000
assert report['probe']['channels'] == 2
assert report['probe']['duration'] >= 10
print(json.dumps(report, indent=2))
display(Audio(report['audio_path']))

## Troubleshooting

- **CUDA OOM:** restart the runtime, set `OFFLOAD_TO_CPU = True`, and keep batch size 1.
- **Hugging Face 401/403:** confirm `HF_TOKEN` can read the private model and Secret access is enabled.
- **OpenRouter 401/402/429:** verify `OPENROUTER_API_KEY`, credits and rate limit; rerun only the enhancer cell.
- **Checksum failure:** delete `/content/melodic-edm-core-v2` and download the same immutable revision again.
- **Missing adapter:** choose exactly `final-all-data` or `best-val`.
- **Initialization failure:** verify both pinned checkpoint downloads completed. Do not substitute another base model.
- **Inference subprocess failure:** open `/content/v2_inference.log` or inspect the final 120 lines printed by the generation cell; `CalledProcessError` alone is only a wrapper, not the underlying error.